## 1

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import RobertaConfig, RobertaForCausalLM, RobertaTokenizer
import Levenshtein
from tqdm import tqdm
import random
import timm
from transformers import RobertaModel
from torch.optim.lr_scheduler import OneCycleLR

In [9]:

BASE_DATA_DIR = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE = os.path.join(BASE_DATA_DIR, "words.txt")

# Set checkpoint path directly in your working project directory
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_1.pth"

# =====================================================================
# 2. DATASET MODULE WITH CORRUPTED IMAGE PROTECTION
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor, transform=None, max_target_length=25):
        self.data_dir = data_dir            
        self.transform = transform
        self.processor = processor          
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"⚠️ Error: Path {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                
                word_id = parts[0]
                status = parts[1]
                if status != "ok":  
                    continue
                
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                id_parts = word_id.split('-')
                folder_grandparent = id_parts[0]                       
                folder_parent = f"{id_parts[0]}-{id_parts[1]}"         
                filename = f"{word_id}.png"                            
                
                img_path = os.path.join(self.data_dir, "words", folder_grandparent, folder_parent, filename)
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Successfully registered {len(self.samples)} valid image-text samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break 
            except Exception as e:
                attempts += 1
                # Silent tracking skip to prevent logging clutter during extended runs
                idx = random.randint(0, len(self.samples) - 1)
        
        if attempts >= len(self.samples):
            raise RuntimeError("Dataset Error: All image items appear corrupted.")

        if self.transform:
            image = self.transform(image)
            
        labels = self.processor(
            text, 
            padding='max_length', 
            max_length=self.max_target_length, 
            truncation=True, 
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        labels = torch.where(labels == self.processor.pad_token_id, torch.tensor(-100), labels)
        return image, labels, text

def get_iam_dataloaders(data_dir, words_txt_path, processor, batch_size=32):
    transforms = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=transforms)
    
    train_size = int(0.9 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader

# =====================================================================
# 3. COMPLETE PIPELINE ARCHITECTURE 
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.copy_(torch.linspace(-1, 1, num_control_points).repeat(2))

    def forward(self, x):
        batch_size = x.size(0)
        points = self.fc(self.conv(x).view(batch_size, -1))
        return points.view(batch_size, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        # Actually use the localization network output
        control_points = self.localization(x)  # (B, K, 2)
        # Use affine approximation from predicted points
        # Take mean shift and scale from control points
        B = x.size(0)
        # Build affine from control point statistics
        cx = control_points[:, :, 0].mean(dim=1)  # (B,)
        cy = control_points[:, :, 1].mean(dim=1)  # (B,)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = cx * 0.1   # small correction
        theta[:, 1, 2] = cy * 0.1
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False,
                             padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        # Swin base stage 4 output is 1024 channels
        self.proj = nn.Linear(1024, d_model)

    def forward(self, x):
        feat = self.swin(x)[-1]      # (B, H', W', 1024)
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)  # (B, seq, 1024)
        return self.proj(feat)       # (B, seq, d_model)

class RoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        # Load pretrained
        roberta = RobertaModel.from_pretrained("roberta-base")
        config  = RobertaConfig.from_pretrained("roberta-base")
        config.is_decoder        = True
        config.add_cross_attention = True
        config.vocab_size        = vocab_size

        self.decoder = RobertaForCausalLM(config)

        # Copy pretrained encoder weights into decoder
        self.decoder.roberta.embeddings.load_state_dict(
            roberta.embeddings.state_dict()
        )
        for i, layer in enumerate(
                self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(
                roberta.encoder.layer[i].state_dict(),
                strict=False   # cross-attn layers won't match
            )
        del roberta

    def forward(self, input_ids, encoder_hidden_states,
                labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )
class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points=num_control_points)
        self.swin_encoder = SwinSimulationEncoder(d_model=d_model)
        
        self.bilstm = nn.LSTM(
            input_size=d_model, 
            hidden_size=d_model // 2, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True
        )
        
        config = RobertaConfig(
            vocab_size=vocab_size,
            hidden_size=d_model,
            num_hidden_layers=4,       
            num_attention_heads=8,
            is_decoder=True,           
            add_cross_attention=True   
        )
        self.roberta_decoder = RobertaForCausalLM(config)
        
    def _extract_visual_memory(self, images):
        rectified_imgs = self.tps_stn(images)
        vis_features = self.swin_encoder(rectified_imgs)  
        
        b, c, h, w = vis_features.size()
        seq_features = vis_features.mean(dim=2).permute(0, 2, 1)  
        
        memory, _ = self.bilstm(seq_features)
        return memory

    def forward(self, images, target_ids):
        memory = self._extract_visual_memory(images)
        clean_target_ids = torch.where(target_ids == -100, torch.tensor(0, device=target_ids.device), target_ids)
        
        outputs = self.roberta_decoder(
            input_ids=clean_target_ids, 
            encoder_hidden_states=memory,
            labels=target_ids
        )
        return outputs.loss

    def generate(self, images, max_length=25, bos_token_id=0, eos_token_id=2):
        self.eval()
        device = images.device
        batch_size = images.size(0)
        
        with torch.no_grad():
            memory = self._extract_visual_memory(images)
            generated = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)
            
            for _ in range(max_length - 1):
                outputs = self.roberta_decoder(input_ids=generated, encoder_hidden_states=memory)
                next_token_logits = outputs.logits[:, -1, :]
                next_tokens = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
                
                generated = torch.cat((generated, next_tokens), dim=1)
                if (generated == eos_token_id).sum(dim=1).min() > 0:
                    break
        return generated

# =====================================================================
# 4. POST-PROCESSING LOGIC AND ERROR METRICS
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = {"move", "to", "stop", "mr.", "gaitskell", "from"}
        if words_txt_file and os.path.exists(words_txt_file):
            self.build_lexicon_from_dataset(words_txt_file)
            
    def build_lexicon_from_dataset(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        self.lexicon.add(word)
        print(f"Agent state tracking verification module populated.")

    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        # Skip correction for numbers, very short words,
        # already-valid words
        if (not cleaned or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
    
        best_candidate = text_output
        min_distance   = float('inf')
    
        # Only check words of similar length — reduces search space
        # from 38k to ~200-300 candidates
        target_len = len(cleaned)
        for legal_word in self.lexicon:
            if abs(len(legal_word) - target_len) > 1:
                continue   # skip if length difference > 1
            dist = Levenshtein.distance(cleaned, legal_word)
            if dist < min_distance and dist == 1:  # only edit-1
                min_distance   = dist
                best_candidate = legal_word
    
        if min_distance == 1:
            return (best_candidate.upper()
                    if text_output.isupper() else best_candidate)
        return text_output

def calculate_metrics(predictions, targets):
    total_char_error_rate = 0.0
    incorrect_words = 0
    total_samples = len(targets)
    
    if total_samples == 0:
        return {"CER": 0.0, "WER": 0.0}

    for pred, target in zip(predictions, targets):
        pred_clean = pred.strip()
        target_clean = target.strip()
        
        edit_dist = Levenshtein.distance(pred_clean, target_clean)
        
        target_len = len(target_clean)
        if target_len > 0:
            total_char_error_rate += (edit_dist / target_len)
        else:
            if len(pred_clean) > 0:
                total_char_error_rate += 1.0 
        
        if edit_dist != 0:
            incorrect_words += 1

    cer = (total_char_error_rate / total_samples) * 100
    wer = (incorrect_words / total_samples) * 100
    return {"CER": cer, "WER": wer}

# =====================================================================
# 5. MULTI-EPOCH PIPELINE WITH FILE PERSISTENCE ARCHIVE
# =====================================================================
def run_training_pipeline(epochs=15, batch_size=32, lr=1e-4):
    print("Pre-fetching RoBERTa WordPiece tokenizers...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    
    train_loader, val_loader = get_iam_dataloaders(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size)
    
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    optimizer = torch.optim.AdamW(
    model.parameters(), lr=5e-5,   # lower LR for pretrained
    weight_decay=0.05              # stronger regularization
    )
    # Add label smoothing to CE loss
    criterion = nn.CrossEntropyLoss(
        ignore_index=-100,
        label_smoothing=0.1
    )
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    # Track the best Word Error Rate achieved to drive disk serialization saves
    best_validation_wer = float('inf')
    
    print(f"\n--- Initiating Model Training Matrix Target: {device} ---")
    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Training]")
        for images, labels, _ in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            loss = model(images, target_ids=labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})
            
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"✅ Epoch {epoch} Complete | Cross-Entropy Loss: {avg_train_loss:.4f}")
        
        # ----------------==================================
        # Full Evaluation Validation Loop Routine
        # ----------------==================================
        model.eval()
        all_predictions = []
        all_targets = []
        
        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch} [Evaluating]"):
                images = images.to(device)
                generated_tokens = model.generate(
                    images, 
                    max_length=25, 
                    bos_token_id=tokenizer.bos_token_id, 
                    eos_token_id=tokenizer.eos_token_id
                )
                
                for idx in range(len(images)):
                    raw_decoded = tokenizer.decode(generated_tokens[idx], skip_special_tokens=True)
                    verified_output = agent_verifier.verify_and_correct(raw_decoded)
                    all_predictions.append(verified_output)
                    all_targets.append(text_labels[idx])

        metrics = calculate_metrics(all_predictions, all_targets)
        current_wer = metrics['WER']
        
        print(f"📊 Epoch {epoch} Validation: CER = {metrics['CER']:.2f}% | WER = {current_wer:.2f}%")
        
        # --- SAFE CHECKPOINT SERIALIZATION GATE ---
        if current_wer < best_validation_wer:
            best_validation_wer = current_wer
            print(f"💾 Validation drop detected! Archiving weights file to disk location: {CHECKPOINT_PATH}")
            
            state_to_save = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_validation_wer,
                'cer': metrics['CER']
            }
            torch.save(state_to_save, CHECKPOINT_PATH)
            
        print("=" * 70)

if __name__ == "__main__":
    # Increased epochs allocation to 15 to permit complex visual cross-attention patterns to settle
    run_training_pipeline(epochs=20, batch_size=32, lr=1e-4)

Pre-fetching RoBERTa WordPiece tokenizers...
Successfully registered 38305 valid image-text samples.
Agent state tracking verification module populated.

--- Initiating Model Training Matrix Target: cuda ---


Epoch 1/20 [Training]: 100%|█████████████████████████████| 1077/1077 [01:13<00:00, 14.74it/s, Loss=2.8658]


✅ Epoch 1 Complete | Cross-Entropy Loss: 3.8382


Epoch 1 [Evaluating]: 100%|█████████████████████████████████████████████| 120/120 [00:04<00:00, 27.07it/s]


📊 Epoch 1 Validation: CER = 68.49% | WER = 75.57%
💾 Validation drop detected! Archiving weights file to disk location: /home/mca24-26/handwritten text detection/iam_htr_1.pth


...

📊 Epoch 19 Validation: CER = 24.41% | WER = 33.18%
💾 Validation drop detected! Archiving weights file to disk location: /home/mca24-26/handwritten text detection/iam_htr_1.pth


Epoch 20/20 [Training]: 100%|████████████████████████████| 1077/1077 [01:13<00:00, 14.74it/s, Loss=0.0287]


✅ Epoch 20 Complete | Cross-Entropy Loss: 0.0540


Epoch 20 [Evaluating]: 100%|████████████████████████████████████████████| 120/120 [00:05<00:00, 23.37it/s]


📊 Epoch 20 Validation: CER = 24.65% | WER = 33.65%


In [1]:

BASE_DATA_DIR = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE = os.path.join(BASE_DATA_DIR, "words.txt")

# Set checkpoint path directly in your working project directory
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_2.pth"

# =====================================================================
# 2. DATASET MODULE WITH CORRUPTED IMAGE PROTECTION
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor, transform=None, max_target_length=25):
        self.data_dir = data_dir            
        self.transform = transform
        self.processor = processor          
        self.max_target_length = max_target_length
        self.samples = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"⚠️ Error: Path {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                
                word_id = parts[0]
                status = parts[1]
                if status != "ok":  
                    continue
                
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                
                id_parts = word_id.split('-')
                folder_grandparent = id_parts[0]                       
                folder_parent = f"{id_parts[0]}-{id_parts[1]}"         
                filename = f"{word_id}.png"                            
                
                img_path = os.path.join(self.data_dir, "words", folder_grandparent, folder_parent, filename)
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Successfully registered {len(self.samples)} valid image-text samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break 
            except Exception as e:
                attempts += 1
                # Silent tracking skip to prevent logging clutter during extended runs
                idx = random.randint(0, len(self.samples) - 1)
        
        if attempts >= len(self.samples):
            raise RuntimeError("Dataset Error: All image items appear corrupted.")

        if self.transform:
            image = self.transform(image)
            
        labels = self.processor(
            text, 
            padding='max_length', 
            max_length=self.max_target_length, 
            truncation=True, 
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        labels = torch.where(labels == self.processor.pad_token_id, torch.tensor(-100), labels)
        return image, labels, text

def get_iam_dataloaders(data_dir, words_txt_path, processor, batch_size=32):
    transforms = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=transforms)
    
    train_size = int(0.9 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader

# =====================================================================
# 3. COMPLETE PIPELINE ARCHITECTURE 
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.copy_(torch.linspace(-1, 1, num_control_points).repeat(2))

    def forward(self, x):
        batch_size = x.size(0)
        points = self.fc(self.conv(x).view(batch_size, -1))
        return points.view(batch_size, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        # Actually use the localization network output
        control_points = self.localization(x)  # (B, K, 2)
        # Use affine approximation from predicted points
        # Take mean shift and scale from control points
        B = x.size(0)
        # Build affine from control point statistics
        cx = control_points[:, :, 0].mean(dim=1)  # (B,)
        cy = control_points[:, :, 1].mean(dim=1)  # (B,)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = cx * 0.1   # small correction
        theta[:, 1, 2] = cy * 0.1
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False,
                             padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        # Swin base stage 4 output is 1024 channels
        self.proj = nn.Linear(1024, d_model)

    def forward(self, x):
        feat = self.swin(x)[-1]      # (B, H', W', 1024)
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)  # (B, seq, 1024)
        return self.proj(feat)       # (B, seq, d_model)

class RoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        # Load pretrained
        roberta = RobertaModel.from_pretrained("roberta-base")
        config  = RobertaConfig.from_pretrained("roberta-base")
        config.is_decoder        = True
        config.add_cross_attention = True
        config.vocab_size        = vocab_size

        self.decoder = RobertaForCausalLM(config)

        # Copy pretrained encoder weights into decoder
        self.decoder.roberta.embeddings.load_state_dict(
            roberta.embeddings.state_dict()
        )
        for i, layer in enumerate(
                self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(
                roberta.encoder.layer[i].state_dict(),
                strict=False   # cross-attn layers won't match
            )
        del roberta

    def forward(self, input_ids, encoder_hidden_states,
                labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )
class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768,
                 num_control_points=16):
        super().__init__()
        self.tps_stn         = TPSSpatialTransformer(num_control_points)
        self.swin_encoder    = SwinEncoder(d_model=d_model)      # NEW class
        self.bilstm          = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.roberta_decoder = RoBERTaDecoder(
            vocab_size=vocab_size, d_model=d_model   # NEW class
        )

    def _extract_visual_memory(self, images):
        rectified  = self.tps_stn(images)         # (B, 3, H, W)
        swin_feat  = self.swin_encoder(rectified)  # (B, seq, 768) — 3D, not 4D
        memory, _  = self.bilstm(swin_feat)        # (B, seq, 768)
        return memory

    def forward(self, images, target_ids, criterion=None):
        memory    = self._extract_visual_memory(images)
        clean_ids = torch.where(
            target_ids == -100,
            torch.zeros_like(target_ids),
            target_ids
        )
        outputs = self.roberta_decoder(
            input_ids             = clean_ids,
            encoder_hidden_states = memory,
            labels                = target_ids
        )
        if criterion is not None:
            return criterion(
                outputs.logits.reshape(-1, outputs.logits.size(-1)),
                target_ids.reshape(-1)
            )
        return outputs.loss

    @torch.no_grad()
    def generate(self, images, max_length=25,
                 bos_token_id=0, eos_token_id=2):
        device    = images.device
        B         = images.size(0)
        memory    = self._extract_visual_memory(images)
        generated = torch.full((B, 1), bos_token_id,
                               dtype=torch.long, device=device)
        for _ in range(max_length - 1):
            outputs     = self.roberta_decoder(
                input_ids             = generated,
                encoder_hidden_states = memory
            )
            next_tokens = outputs.logits[:, -1, :].argmax(
                dim=-1, keepdim=True
            )
            generated = torch.cat([generated, next_tokens], dim=1)
            if (generated == eos_token_id).any(dim=1).all():
                break
        return generated
# =====================================================================
# 4. POST-PROCESSING LOGIC AND ERROR METRICS
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = {"move", "to", "stop", "mr.", "gaitskell", "from"}
        if words_txt_file and os.path.exists(words_txt_file):
            self.build_lexicon_from_dataset(words_txt_file)
            
    def build_lexicon_from_dataset(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        self.lexicon.add(word)
        print(f"Agent state tracking verification module populated.")

    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        # Skip correction for numbers, very short words,
        # already-valid words
        if (not cleaned or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
    
        best_candidate = text_output
        min_distance   = float('inf')
    
        # Only check words of similar length — reduces search space
        # from 38k to ~200-300 candidates
        target_len = len(cleaned)
        for legal_word in self.lexicon:
            if abs(len(legal_word) - target_len) > 1:
                continue   # skip if length difference > 1
            dist = Levenshtein.distance(cleaned, legal_word)
            if dist < min_distance and dist == 1:  # only edit-1
                min_distance   = dist
                best_candidate = legal_word
    
        if min_distance == 1:
            return (best_candidate.upper()
                    if text_output.isupper() else best_candidate)
        return text_output

def calculate_metrics(predictions, targets):
    total_char_error_rate = 0.0
    incorrect_words = 0
    total_samples = len(targets)
    
    if total_samples == 0:
        return {"CER": 0.0, "WER": 0.0}

    for pred, target in zip(predictions, targets):
        pred_clean = pred.strip()
        target_clean = target.strip()
        
        edit_dist = Levenshtein.distance(pred_clean, target_clean)
        
        target_len = len(target_clean)
        if target_len > 0:
            total_char_error_rate += (edit_dist / target_len)
        else:
            if len(pred_clean) > 0:
                total_char_error_rate += 1.0 
        
        if edit_dist != 0:
            incorrect_words += 1

    cer = (total_char_error_rate / total_samples) * 100
    wer = (incorrect_words / total_samples) * 100
    return {"CER": cer, "WER": wer}

# =====================================================================
# 5. MULTI-EPOCH PIPELINE WITH FILE PERSISTENCE ARCHIVE
# =====================================================================
def run_training_pipeline(epochs=15, batch_size=32, lr=1e-4):
    print("Pre-fetching RoBERTa WordPiece tokenizers...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    
    train_loader, val_loader = get_iam_dataloaders(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size)
    
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    optimizer = torch.optim.AdamW(
    model.parameters(), lr=5e-5,   # lower LR for pretrained
    weight_decay=0.05              # stronger regularization
    )
    # Add label smoothing to CE loss
    criterion = nn.CrossEntropyLoss(
        ignore_index=-100,
        label_smoothing=0.1
    )
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    # Track the best Word Error Rate achieved to drive disk serialization saves
    best_validation_wer = float('inf')
    
    print(f"\n--- Initiating Model Training Matrix Target: {device} ---")
    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Training]")
        for images, labels, _ in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            loss = model(images, target_ids=labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})
            
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"✅ Epoch {epoch} Complete | Cross-Entropy Loss: {avg_train_loss:.4f}")
        
        # ----------------==================================
        # Full Evaluation Validation Loop Routine
        # ----------------==================================
        model.eval()
        all_predictions = []
        all_targets = []
        
        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch} [Evaluating]"):
                images = images.to(device)
                generated_tokens = model.generate(
                    images, 
                    max_length=25, 
                    bos_token_id=tokenizer.bos_token_id, 
                    eos_token_id=tokenizer.eos_token_id
                )
                
                for idx in range(len(images)):
                    raw_decoded = tokenizer.decode(generated_tokens[idx], skip_special_tokens=True)
                    verified_output = agent_verifier.verify_and_correct(raw_decoded)
                    all_predictions.append(verified_output)
                    all_targets.append(text_labels[idx])

        metrics = calculate_metrics(all_predictions, all_targets)
        current_wer = metrics['WER']
        
        print(f"📊 Epoch {epoch} Validation: CER = {metrics['CER']:.2f}% | WER = {current_wer:.2f}%")
        
        # --- SAFE CHECKPOINT SERIALIZATION GATE ---
        if current_wer < best_validation_wer:
            best_validation_wer = current_wer
            print(f"💾 Validation drop detected! Archiving weights file to disk location: {CHECKPOINT_PATH}")
            
            state_to_save = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_validation_wer,
                'cer': metrics['CER']
            }
            torch.save(state_to_save, CHECKPOINT_PATH)
            
        print("=" * 70)

if __name__ == "__main__":
    # Increased epochs allocation to 15 to permit complex visual cross-attention patterns to settle
    run_training_pipeline(epochs=20, batch_size=32, lr=1e-4)

Pre-fetching RoBERTa WordPiece tokenizers...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Successfully registered 38305 valid image-text samples.


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

...

📊 Epoch 19 Validation: CER = 15.86% | WER = 23.75%
💾 Validation drop detected! Archiving weights file to disk location: /home/mca24-26/handwritten text detection/iam_htr_2.pth


Epoch 20/20 [Training]: 100%|████████████████████████████| 1077/1077 [03:37<00:00,  4.95it/s, Loss=0.0261]


✅ Epoch 20 Complete | Cross-Entropy Loss: 0.0625


Epoch 20 [Evaluating]: 100%|████████████████████████████████████████████| 120/120 [00:12<00:00,  9.74it/s]

📊 Epoch 20 Validation: CER = 17.18% | WER = 25.16%


In [4]:
# test
import os
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer
from tqdm import tqdm

def run_notebook_evaluation(num_preview_samples=15):
    print("🤖 Initializing HTR Notebook Test Engine...")
    
    # 1. Fetch tokenizer
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    
    # 2. Extract Data Split (Re-uses your validation set loader)
    print("📂 Mapping validation/test sequence streams...")
    _, test_loader = get_iam_dataloaders(
        BASE_DATA_DIR, 
        WORDS_TXT_FILE, 
        processor=tokenizer, 
        batch_size=32
    )
    
    # 3. Rebuild Complete Pipeline Architecture Matrix
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # 4. Checkpoint Serialization Retrieval
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"❌ Missing checkpoint file at: {CHECKPOINT_PATH}")
        print("Please ensure your training loop successfully executed and saved 'iam_htr_2.pth'.")
        return
        
    print(f"💾 Restoring weights from disk target: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ State loaded! Recovered from Epoch {checkpoint['epoch']} (Best WER: {checkpoint.get('best_wer', 0.0):.2f}%)")
    
    # 5. Initialize Post-processing Agent
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    # 6. Evaluation Routine Pass
    model.eval()
    all_predictions = []
    all_targets = []
    preview_log = []

    print("\n--- Running Inference Matrix over Evaluation Loader ---")
    with torch.no_grad():
        for images, _, text_labels in tqdm(test_loader, desc="[Evaluating Model]"):
            images = images.to(device)
            
            # Autoregressive generation pass
            generated_tokens = model.generate(
                images, 
                max_length=25, 
                bos_token_id=tokenizer.bos_token_id, 
                eos_token_id=tokenizer.eos_token_id
            )
            
            for idx in range(len(images)):
                raw_decoded = tokenizer.decode(generated_tokens[idx], skip_special_tokens=True)
                verified_output = agent_verifier.verify_and_correct(raw_decoded)
                
                all_predictions.append(verified_output)
                all_targets.append(text_labels[idx])
                
                # Dynamic sample capture for visualization 
                if len(preview_log) < num_preview_samples:
                    preview_log.append({
                        "Ground Truth": text_labels[idx],
                        "Raw Model Prediction": raw_decoded.strip(),
                        "Agent Corrected": verified_output.strip()
                    })

    # 7. Calculate error rates using your built-in Levenshtein metrics
    metrics = calculate_metrics(all_predictions, all_targets)
    
    print("\n" + "="*80)
    print("📊 FINAL COGNITIVE ERROR METRICS")
    print("="*80)
    print(f"▶️ CHARACTER ERROR RATE (CER) : {metrics['CER']:.2f}%")
    print(f"▶️ WORD ERROR RATE (WER)      : {metrics['WER']:.2f}%")
    print("-" * 80)
    
    # 8. Render Results Table
    print(f"\n📋 SAMPLE INFERENCE RECONCILIATION ({num_preview_samples} Samples):")
    print(f"{'GROUND TRUTH':<25} | {'RAW PREDICTION':<25} | {'AGENT CORRECTED':<25}")
    print("-" * 81)
    for log in preview_log:
        print(f"{log['Ground Truth']:<25} | {log['Raw Model Prediction']:<25} | {log['Agent Corrected']:<25}")
    print("=" * 80 + "\n")

# Run evaluation directly in your notebook workspace
run_notebook_evaluation(num_preview_samples=15)

🤖 Initializing HTR Notebook Test Engine...
📂 Mapping validation/test sequence streams...
Successfully registered 38305 valid image-text samples.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


💾 Restoring weights from disk target: /home/mca24-26/handwritten text detection/iam_htr_2.pth


/tmp/ipykernel_3236132/1752798426.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)


✅ State loaded! Recovered from Epoch 19 (Best WER: 23.75%)
Agent state tracking verification module populated.

--- Running Inference Matrix over Evaluation Loader ---


[Evaluating Model]: 100%|███████████████████████████████████████████████| 120/120 [00:12<00:00,  9.88it/s]


📊 FINAL COGNITIVE ERROR METRICS
▶️ CHARACTER ERROR RATE (CER) : 3.96%
▶️ WORD ERROR RATE (WER)      : 5.69%
--------------------------------------------------------------------------------

📋 SAMPLE INFERENCE RECONCILIATION (15 Samples):
GROUND TRUTH              | RAW PREDICTION            | AGENT CORRECTED          
---------------------------------------------------------------------------------
we                        | we                        | we                       
her                       | her                       | her                      
find                      | find                      | find                     
and                       | and                       | and                      
read                      | read                      | read                     
to                        | to                        | to                       
.                         | .                         | .                        
Couve                  

## 2 (fix: increase input res, llrd, strong agentic verification, cosine ann, beam search)

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, RobertaModel,
                          RobertaConfig, RobertaForCausalLM)
import timm
import Levenshtein
from tqdm import tqdm
import math

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_4.pth"

# =====================================================================
# 2. CONFIG
# =====================================================================
IMG_HEIGHT      = 96    
IMG_WIDTH       = 384   
MAX_SEQ_LEN     = 25
BEAM_SIZE       = 5
D_MODEL         = 768

# =====================================================================
# 3. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 transform=None, max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images corrupted.")
        if self.transform:
            image = self.transform(image)
        labels = self.processor(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        labels = torch.where(
            labels == self.processor.pad_token_id,
            torch.tensor(-100), labels
        )
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=None)
    total      = len(full_dataset)
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(42)
    indices    = torch.randperm(total, generator=generator).tolist()

    train_samples = [full_dataset.samples[i] for i in indices[:train_size]]
    val_samples   = [full_dataset.samples[i] for i in indices[train_size:]]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=MAX_SEQ_LEN):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_loader = DataLoader(
        SplitDataset(train_samples, processor, train_transform),
        batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True
    )
    val_loader = DataLoader(
        SplitDataset(val_samples, processor, val_transform),
        batch_size=batch_size, shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 4. ARCHITECTURE
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B   = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        cp             = self.localization(x)
        cx             = cp[:, :, 0].mean(dim=1)
        cy             = cp[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))  # (B, seq, d_model)


class RoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        roberta = RobertaModel.from_pretrained("roberta-base")
        config  = RobertaConfig.from_pretrained("roberta-base")
        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size
        self.decoder = RobertaForCausalLM(config)
        self.decoder.roberta.embeddings.load_state_dict(
            roberta.embeddings.state_dict()
        )
        for i, layer in enumerate(self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(
                roberta.encoder.layer[i].state_dict(),
                strict=False
            )
        del roberta

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        self.tps_stn         = TPSSpatialTransformer(num_control_points)
        self.swin_encoder    = SwinEncoder(d_model=d_model)
        self.bilstm          = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.roberta_decoder = RoBERTaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),
            dec_input
        )
        labels = target_ids[:, 1:].clone()

        outputs = self.roberta_decoder(
            input_ids             = dec_input,
            encoder_hidden_states = memory,
            labels                = None
        )
        logits = outputs.logits

        if criterion is not None:
            return criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100
        )

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2, beam_size=BEAM_SIZE):
        """ Beam search decoding script (Preserved for high-accuracy test benchmarks) """
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)

        all_results = []

        for b in range(B):
            mem       = memory[b:b+1]             # (1, seq, d)
            beams     = [(0.0, [bos_token_id])]   # (score, tokens)
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor([tokens], dtype=torch.long, device=device)
                    out = self.roberta_decoder(
                        input_ids             = inp,
                        encoder_hidden_states = mem,
                        labels                = None
                    )
                    log_prob = F.log_softmax(out.logits[0, -1], dim=-1)
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                        candidates.append((score + lp, tokens + [tid]))

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams         = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break

                if not beams:
                    break

            all_beams  = completed if completed else beams
            best_score, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded  = torch.full((B, max_len), eos_token_id, dtype=torch.long, device=device)
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded

    @torch.no_grad()
    def generate_greedy(self, images, max_length=MAX_SEQ_LEN, bos_token_id=0, eos_token_id=2):
        """ Fully-vectorized batch greedy generation for super-fast training validation """
        device = images.device
        B = images.size(0)
        
        # 1. Extract sequential feature memory map once
        memory = self._extract_visual_memory(images)
        
        # 2. Init target matrix structure (Shape: B x 1)
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        
        for _ in range(max_length - 1):
            outputs = self.roberta_decoder(
                input_ids=generated, 
                encoder_hidden_states=memory
            )
            
            # Predict only against the newest timestamp sequence block
            next_token_logits = outputs.logits[:, -1, :]
            next_tokens = next_token_logits.argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tokens], dim=1)
            
            # Update sequence termination map status
            finished |= (next_tokens.squeeze(-1) == eos_token_id)
            if finished.all():
                break
                
        return generated[:, 1:] # Drop starting BOS token row


# =====================================================================
# 5. IMPROVED AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon   = {}   
        self.freq_max  = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = {}
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] = freq.get(word, 0) + 1
        self.lexicon  = freq
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon built: {len(self.lexicon)} unique words.")

    def _normalized_freq(self, word):
        return self.lexicon.get(word, 0) / self.freq_max

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.7):
        cleaned = text_output.strip().lower()

        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if confidence is not None and confidence >= confidence_threshold:
            return text_output

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue

            freq_score = self._normalized_freq(word)
            score      = freq_score - (dist * 0.5)

            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        if text_output.isupper():
            return best_candidate.upper()
        elif text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# 6. METRICS
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 7. LLRD OPTIMIZER
# =====================================================================
def build_llrd_optimizer(model, base_lr=5e-5, decay_factor=0.75, weight_decay=0.05):
    lr0 = base_lr
    lr1 = base_lr * decay_factor
    lr2 = base_lr * (decay_factor ** 2)
    lr3 = base_lr * (decay_factor ** 3)
    lr4 = base_lr * (decay_factor ** 4)
    lr5 = base_lr * (decay_factor ** 5)
    lr6 = base_lr * (decay_factor ** 6)

    assigned = set()

    decoder_params = []
    for name, param in model.roberta_decoder.named_parameters():
        if id(param) not in assigned:
            decoder_params.append(param)
            assigned.add(id(param))

    bilstm_params = []
    for name, param in model.bilstm.named_parameters():
        if id(param) not in assigned:
            bilstm_params.append(param)
            assigned.add(id(param))

    swin_proj_params = []
    for name, param in model.swin_encoder.named_parameters():
        if id(param) not in assigned and (name.startswith("proj.") or name.startswith("norm.")):
            swin_proj_params.append(param)
            assigned.add(id(param))

    swin_s4_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_3" in name:
            swin_s4_params.append(param)
            assigned.add(id(param))

    swin_s3_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_2" in name:
            swin_s3_params.append(param)
            assigned.add(id(param))

    swin_s2_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_1" in name:
            swin_s2_params.append(param)
            assigned.add(id(param))

    swin_s1_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned:
            swin_s1_params.append(param)
            assigned.add(id(param))

    tps_params = []
    for name, param in model.tps_stn.named_parameters():
        if id(param) not in assigned:
            tps_params.append(param)
            assigned.add(id(param))

    param_groups = [
        {"params": decoder_params,   "lr": lr0, "name": "roberta_decoder"},
        {"params": bilstm_params,    "lr": lr1, "name": "bilstm"},
        {"params": swin_proj_params, "lr": lr2, "name": "swin_proj"},
        {"params": swin_s4_params,   "lr": lr3, "name": "swin_stage4"},
        {"params": swin_s3_params,   "lr": lr4, "name": "swin_stage3"},
        {"params": swin_s2_params,   "lr": lr5, "name": "swin_stage2"},
        {"params": swin_s1_params,   "lr": lr6, "name": "swin_stage1_embed"},
        {"params": tps_params,       "lr": lr2, "name": "tps_stn"},
    ]

    param_groups = [g for g in param_groups if len(g["params"]) > 0]

    print("\nLLRD Optimizer Groups:")
    print(f"{'Group':<22} {'LR':>10} {'Params':>10}")
    print("-" * 47)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<22} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)
    return optimizer


# =====================================================================
# 8. TRAINING
# =====================================================================
def run_training_pipeline(epochs=50, batch_size=32, base_lr=5e-5,
                         early_stopping_patience=8, resume_from=None):

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id} | PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total_p/1e6:.2f}M")
    print(f"Trainable params: {trainable_p/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

    optimizer = build_llrd_optimizer(model, base_lr=base_lr, decay_factor=0.75, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)

    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    early_stopper  = EarlyStopping(patience=early_stopping_patience, min_delta=0.05)
    best_val_wer   = float('inf')
    start_epoch    = 1

    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        print(f"Resumed at epoch {start_epoch} | Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")
    print(f"Input resolution: {IMG_HEIGHT}x{IMG_WIDTH}")

    for epoch in range(start_epoch, epochs + 1):
        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")

        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels, criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        lrs = [f"{g['lr']:.2e}" for g in optimizer.param_groups]
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | LRs: decoder={lrs[0]} bilstm={lrs[1]}")

        # ── Validate (Optimized via Vectorized Greedy Decoding) ──
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                
                # FIXED: Swapped to generate_greedy for accelerated parallel batches
                tokens = model.generate_greedy(
                    images,
                    max_length   = MAX_SEQ_LEN,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id
                )

                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                        print(f"  Target: '{text_labels[i]}' | Pred: '{raw.strip()}'")
                    print("---")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(tokens[i], skip_special_tokens=True)
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            print(f"New Best WER achieved! Saving configuration parameters...")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_val_wer,
            }, CHECKPOINT_PATH)

        if early_stopper(current_wer):
            print("Early stopping triggered. Halting run execution.")
            break

if __name__ == "__main__":
    run_training_pipeline(epochs=50, batch_size=32)

Loading tokenizer...
BOS=0 | EOS=2 | PAD=1
Registered 38305 samples.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total params    : 247.90M
Trainable params: 247.90M

LLRD Optimizer Groups:
Group                          LR     Params
-----------------------------------------------
roberta_decoder          5.00e-05    153.06M
bilstm                   3.75e-05      7.09M
swin_proj                2.81e-05      0.40M
swin_stage4              2.11e-05     27.30M
swin_stage3              1.58e-05     57.31M
swin_stage2              1.19e-05      1.71M
swin_stage1_embed        8.90e-06      0.40M
tps_stn                  2.81e-05      0.63M
Lexicon built: 6231 unique words.

Training on: cuda
Input resolution: 96x384


Epoch 1/50 [Train]: 100%|████████████████████████████████| 1077/1077 [11:08<00:00,  1.61it/s, loss=3.9368]


Epoch 1 | Loss: 4.5340 | LRs: decoder=4.88e-05 bilstm=3.66e-05


...

Epoch 37 | Loss: 1.4955 | LRs: decoder=4.63e-05 bilstm=3.47e-05


Epoch 37/50 [Val]:   2%|▊                                                 | 2/120 [00:00<00:26,  4.43it/s]


--- Debug Epoch 37 ---
  Target: 'any' | Pred: 'any'
  Target: 'any' | Pred: 'any'
  Target: 'unopposed' | Pred: 'mapposed'
---


Epoch 37/50 [Val]: 100%|████████████████████████████████████████████████| 120/120 [00:18<00:00,  6.36it/s]

Epoch 37 | CER: 10.26% | WER: 18.53%
EarlyStopping: 8/8
Early stopping triggered. Halting run execution.


In [3]:
#test
# =====================================================================
# 9. EVALUATION & BENCHMARKING (BEAM SEARCH)
# =====================================================================
import torch
from tqdm import tqdm
from transformers import RobertaTokenizer

def run_testing_pipeline(checkpoint_path=CHECKPOINT_PATH, beam_size=BEAM_SIZE, batch_size=16):
    """
    Evaluates the trained CompleteHTRPipeline model using Beam Search decoding.
    Note: Lowered batch_size to 16 default because beam search is memory-intensive.
    """
    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    
    # 1. Get the data loaders (We use the val_loader as our test set split here)
    print("Sourcing dataset splits...")
    _, test_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing on device: {device}")
    
    # 2. Initialize Model Architecture
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    
    # 3. Load Weights
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"No checkpoint found at {checkpoint_path}. Ensure you have trained weights saved.")
        
    print(f"Loading weights from checkpoint: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded weights from Epoch {ckpt.get('epoch', 'N/A')} (Best Training Validation WER: {ckpt.get('best_wer', float('inf')):.2f}%)")
    
    # 4. Initialize Post-Processing Agents
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    # 5. Evaluation Loop
    model.eval()
    all_raw_preds = []
    all_verified_preds = []
    all_targets = []
    
    print(f"\nRunning Inference via Beam Search (Beam Size = {beam_size})...")
    with torch.no_grad():
        for images, _, text_labels in tqdm(test_loader, desc="[Testing Beam Search]"):
            images = images.to(device)
            
            # Invoking your custom loop-based Beam Search generation matrix
            tokens = model.generate(
                images,
                max_length=MAX_SEQ_LEN,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                beam_size=beam_size
            )
            
            # Decode and verify predictions
            for i in range(images.size(0)):
                raw_pred = tokenizer.decode(tokens[i], skip_special_tokens=True).strip()
                verified_pred = agent_verifier.verify_and_correct(raw_pred)
                
                all_raw_preds.append(raw_pred)
                all_verified_preds.append(verified_pred)
                all_targets.append(text_labels[i])

    # 6. Calculate and Display Comparative Metrics
    raw_metrics = calculate_metrics(all_raw_preds, all_targets)
    verified_metrics = calculate_metrics(all_verified_preds, all_targets)
    
    print("\n" + "="*50)
    print(f"{'FINAL HTR BENCHMARK REPORT (BEAM SEARCH)':^50}")
    print("="*50)
    print(f"{'Metric':<25} | {'Raw Model Output':<18} | {'Agent Verified Output'}")
    print("-"*50)
    print(f"{'Character Error Rate (CER)':<25} | {raw_metrics['CER']:>15}% | {verified_metrics['CER']:>20}%")
    print(f"{'Word Error Rate (WER)':<25} | {raw_metrics['WER']:>15}% | {verified_metrics['WER']:>20}%")
    print("="*50)
    
    # 7. Print Sample Qualitative Outputs
    print("\n--- Random Sample Predictions ---")
    sample_indices = random.sample(range(len(all_targets)), min(5, len(all_targets)))
    for idx in sample_indices:
        print(f"Target Ground Truth : '{all_targets[idx]}'")
        print(f"Raw Beam Search Output: '{all_raw_preds[idx]}'")
        print(f"Agentic Verified    : '{all_verified_preds[idx]}'")
        print("-" * 35)

# =====================================================================
# EXECUTION ENTRY POINT
# =====================================================================
# Adjust beam size or batch size here depending on your available GPU VRAM.
run_testing_pipeline(beam_size=5, batch_size=16)

Loading tokenizer...
Sourcing dataset splits...
Registered 38305 samples.
Testing on device: cuda


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading weights from checkpoint: /home/mca24-26/handwritten text detection/iam_htr_4.pth


/tmp/ipykernel_4083772/643022325.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=device)


Loaded weights from Epoch 29 (Best Training Validation WER: 17.02%)
Lexicon built: 6231 unique words.

Running Inference via Beam Search (Beam Size = 5)...


[Testing Beam Search]: 100%|██████████████████████████████████████████| 240/240 [1:27:01<00:00, 21.75s/it]


     FINAL HTR BENCHMARK REPORT (BEAM SEARCH)     
Metric                    | Raw Model Output   | Agent Verified Output
--------------------------------------------------
Character Error Rate (CER) |           9.407% |               9.5022%
Word Error Rate (WER)     |         17.0713% |               16.758%

--- Random Sample Predictions ---
Target Ground Truth : 'loop'
Raw Beam Search Output: 'loop'
Agentic Verified    : 'loop'
-----------------------------------
Target Ground Truth : 'say'
Raw Beam Search Output: 'say'
Agentic Verified    : 'say'
-----------------------------------
Target Ground Truth : 'The'
Raw Beam Search Output: 'The'
Agentic Verified    : 'The'
-----------------------------------
Target Ground Truth : ','
Raw Beam Search Output: ','
Agentic Verified    : ','
-----------------------------------
Target Ground Truth : 'quit'
Raw Beam Search Output: 'quit'
Agentic Verified    : 'quit'
-----------------------------------


## 3

In [5]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, RobertaModel,
                          RobertaConfig, RobertaForCausalLM)
import timm
import Levenshtein
from tqdm import tqdm
import math

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_5.pth"

# =====================================================================
# 2. CONFIG
# =====================================================================
IMG_HEIGHT      = 96    
IMG_WIDTH       = 384   
MAX_SEQ_LEN     = 25
BEAM_SIZE       = 5
D_MODEL         = 768

# =====================================================================
# 3. DATASET
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 transform=None, max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images corrupted.")
        if self.transform:
            image = self.transform(image)
        labels = self.processor(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        labels = torch.where(
            labels == self.processor.pad_token_id,
            torch.tensor(-100), labels
        )
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=None)
    total      = len(full_dataset)
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(42)
    indices    = torch.randperm(total, generator=generator).tolist()

    train_samples = [full_dataset.samples[i] for i in indices[:train_size]]
    val_samples   = [full_dataset.samples[i] for i in indices[train_size:]]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=MAX_SEQ_LEN):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_loader = DataLoader(
        SplitDataset(train_samples, processor, train_transform),
        batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True
    )
    val_loader = DataLoader(
        SplitDataset(val_samples, processor, val_transform),
        batch_size=batch_size, shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 4. ARCHITECTURE
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B   = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        cp             = self.localization(x)
        cx             = cp[:, :, 0].mean(dim=1)
        cy             = cp[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))  # (B, seq, d_model)


class RoBERTaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        roberta = RobertaModel.from_pretrained("roberta-base")
        config  = RobertaConfig.from_pretrained("roberta-base")
        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size
        self.decoder = RobertaForCausalLM(config)
        self.decoder.roberta.embeddings.load_state_dict(
            roberta.embeddings.state_dict()
        )
        for i, layer in enumerate(self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(
                roberta.encoder.layer[i].state_dict(),
                strict=False
            )
        del roberta

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        self.tps_stn         = TPSSpatialTransformer(num_control_points)
        self.swin_encoder    = SwinEncoder(d_model=d_model)
        self.bilstm          = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.roberta_decoder = RoBERTaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),
            dec_input
        )
        labels = target_ids[:, 1:].clone()

        outputs = self.roberta_decoder(
            input_ids             = dec_input,
            encoder_hidden_states = memory,
            labels                = None
        )
        logits = outputs.logits

        if criterion is not None:
            return criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100
        )

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2, beam_size=BEAM_SIZE):
        """ Beam search decoding script (Preserved for high-accuracy test benchmarks) """
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)

        all_results = []

        for b in range(B):
            mem       = memory[b:b+1]             # (1, seq, d)
            beams     = [(0.0, [bos_token_id])]   # (score, tokens)
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor([tokens], dtype=torch.long, device=device)
                    out = self.roberta_decoder(
                        input_ids             = inp,
                        encoder_hidden_states = mem,
                        labels                = None
                    )
                    log_prob = F.log_softmax(out.logits[0, -1], dim=-1)
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                        candidates.append((score + lp, tokens + [tid]))

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams         = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break

                if not beams:
                    break

            all_beams  = completed if completed else beams
            best_score, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded  = torch.full((B, max_len), eos_token_id, dtype=torch.long, device=device)
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded

    @torch.no_grad()
    def generate_greedy(self, images, max_length=MAX_SEQ_LEN, bos_token_id=0, eos_token_id=2):
        """ Fully-vectorized batch greedy generation for super-fast training validation """
        device = images.device
        B = images.size(0)
        
        # 1. Extract sequential feature memory map once
        memory = self._extract_visual_memory(images)
        
        # 2. Init target matrix structure (Shape: B x 1)
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        
        for _ in range(max_length - 1):
            outputs = self.roberta_decoder(
                input_ids=generated, 
                encoder_hidden_states=memory
            )
            
            # Predict only against the newest timestamp sequence block
            next_token_logits = outputs.logits[:, -1, :]
            next_tokens = next_token_logits.argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tokens], dim=1)
            
            # Update sequence termination map status
            finished |= (next_tokens.squeeze(-1) == eos_token_id)
            if finished.all():
                break
                
        return generated[:, 1:] # Drop starting BOS token row


# =====================================================================
# 5. IMPROVED AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon   = {}   
        self.freq_max  = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = {}
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] = freq.get(word, 0) + 1
        self.lexicon  = freq
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon built: {len(self.lexicon)} unique words.")

    def _normalized_freq(self, word):
        return self.lexicon.get(word, 0) / self.freq_max

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.7):
        cleaned = text_output.strip().lower()

        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if confidence is not None and confidence >= confidence_threshold:
            return text_output

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue

            freq_score = self._normalized_freq(word)
            score      = freq_score - (dist * 0.5)

            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        if text_output.isupper():
            return best_candidate.upper()
        elif text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# 6. METRICS
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 7. LLRD OPTIMIZER
# =====================================================================
def build_llrd_optimizer(model, base_lr=5e-5, decay_factor=0.75, weight_decay=0.05):
    lr0 = base_lr
    lr1 = base_lr * decay_factor
    lr2 = base_lr * (decay_factor ** 2)
    lr3 = base_lr * (decay_factor ** 3)
    lr4 = base_lr * (decay_factor ** 4)
    lr5 = base_lr * (decay_factor ** 5)
    lr6 = base_lr * (decay_factor ** 6)

    assigned = set()

    decoder_params = []
    for name, param in model.roberta_decoder.named_parameters():
        if id(param) not in assigned:
            decoder_params.append(param)
            assigned.add(id(param))

    bilstm_params = []
    for name, param in model.bilstm.named_parameters():
        if id(param) not in assigned:
            bilstm_params.append(param)
            assigned.add(id(param))

    swin_proj_params = []
    for name, param in model.swin_encoder.named_parameters():
        if id(param) not in assigned and (name.startswith("proj.") or name.startswith("norm.")):
            swin_proj_params.append(param)
            assigned.add(id(param))

    swin_s4_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_3" in name:
            swin_s4_params.append(param)
            assigned.add(id(param))

    swin_s3_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_2" in name:
            swin_s3_params.append(param)
            assigned.add(id(param))

    swin_s2_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_1" in name:
            swin_s2_params.append(param)
            assigned.add(id(param))

    swin_s1_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned:
            swin_s1_params.append(param)
            assigned.add(id(param))

    tps_params = []
    for name, param in model.tps_stn.named_parameters():
        if id(param) not in assigned:
            tps_params.append(param)
            assigned.add(id(param))

    param_groups = [
        {"params": decoder_params,   "lr": lr0, "name": "roberta_decoder"},
        {"params": bilstm_params,    "lr": lr1, "name": "bilstm"},
        {"params": swin_proj_params, "lr": lr2, "name": "swin_proj"},
        {"params": swin_s4_params,   "lr": lr3, "name": "swin_stage4"},
        {"params": swin_s3_params,   "lr": lr4, "name": "swin_stage3"},
        {"params": swin_s2_params,   "lr": lr5, "name": "swin_stage2"},
        {"params": swin_s1_params,   "lr": lr6, "name": "swin_stage1_embed"},
        {"params": tps_params,       "lr": lr2, "name": "tps_stn"},
    ]

    param_groups = [g for g in param_groups if len(g["params"]) > 0]

    print("\nLLRD Optimizer Groups:")
    print(f"{'Group':<22} {'LR':>10} {'Params':>10}")
    print("-" * 47)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<22} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)
    return optimizer


# =====================================================================
# 8. TRAINING
# =====================================================================
def run_training_pipeline(epochs=50, batch_size=32, base_lr=5e-5,
                         early_stopping_patience=8, resume_from=None):

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id} | PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total_p/1e6:.2f}M")
    print(f"Trainable params: {trainable_p/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)

    optimizer = build_llrd_optimizer(model, base_lr=base_lr, decay_factor=0.75, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max  = epochs,   # decay over full training run
    eta_min = 1e-7
)

    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    early_stopper  = EarlyStopping(patience  = 12,min_delta = 0.1)
    best_val_wer   = float('inf')
    start_epoch    = 1

    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        print(f"Resumed at epoch {start_epoch} | Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")
    print(f"Input resolution: {IMG_HEIGHT}x{IMG_WIDTH}")

    for epoch in range(start_epoch, epochs + 1):
        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")

        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels, criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        lrs = [f"{g['lr']:.2e}" for g in optimizer.param_groups]
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | LRs: decoder={lrs[0]} bilstm={lrs[1]}")

        # ── Validate (Optimized via Vectorized Greedy Decoding) ──
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                
                # FIXED: Swapped to generate_greedy for accelerated parallel batches
                tokens = model.generate_greedy(
                    images,
                    max_length   = MAX_SEQ_LEN,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id
                )

                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                        print(f"  Target: '{text_labels[i]}' | Pred: '{raw.strip()}'")
                    print("---")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(tokens[i], skip_special_tokens=True)
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            print(f"New Best WER achieved! Saving configuration parameters...")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_val_wer,
            }, CHECKPOINT_PATH)

        if early_stopper(current_wer):
            print("Early stopping triggered. Halting run execution.")
            break

# if __name__ == "__main__":
#     run_training_pipeline(epochs=50, batch_size=32)

In [6]:
# =====================================================================
# TESTING SCRIPT — RoBERTa HTR Pipeline
# Run this in a separate cell after training is complete
# All architecture classes must be defined above (from training script)
# =====================================================================

import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer
from PIL import Image
import torchvision.transforms as T
import Levenshtein
from tqdm import tqdm
from collections import defaultdict

# =====================================================================
# TEST CONFIG — must match training exactly
# =====================================================================
TEST_CHECKPOINT = "/home/mca24-26/handwritten text detection/iam_htr_5.pth"
TEST_BEAM_SIZE  = 5     # beam search for best accuracy at test time
TEST_BATCH_SIZE = 16
SHOW_SAMPLES    = 20
SAVE_RESULTS    = True

# =====================================================================
# TEST DATASET
# Replicates exact val split from training using seed=42
# Guarantees zero overlap with training data
# =====================================================================
class TestOnlyDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)
        if self.transform:
            image = self.transform(image)
        return image, text


def get_test_loader(batch_size=TEST_BATCH_SIZE):
    transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    # Parse all samples
    all_samples = []
    with open(WORDS_TXT_FILE, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) < 9 or parts[1] != "ok":
                continue
            transcription = " ".join(parts[8:])
            if not transcription.strip():
                continue
            word_id  = parts[0]
            id_parts = word_id.split('-')
            img_path = os.path.join(
                BASE_DATA_DIR, "words",
                id_parts[0],
                f"{id_parts[0]}-{id_parts[1]}",
                f"{word_id}.png"
            )
            if os.path.exists(img_path):
                all_samples.append((img_path, transcription))

    total      = len(all_samples)
    train_size = int(0.9 * total)

    # CRITICAL: same seed=42 as training
    generator   = torch.Generator().manual_seed(42)
    indices     = torch.randperm(total, generator=generator).tolist()
    val_indices = indices[train_size:]
    val_samples = [all_samples[i] for i in val_indices]

    print(f"Total dataset  : {total} samples")
    print(f"Train split    : {train_size} samples")
    print(f"Test split     : {len(val_samples)} samples")

    loader = DataLoader(
        TestOnlyDataset(val_samples, transform),
        batch_size  = batch_size,
        shuffle     = False,
        num_workers = 4
    )
    return loader, val_samples


# =====================================================================
# METRICS
# =====================================================================
def test_calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0, "ExactMatch": 0.0}
    total_cer    = 0.0
    wrong_words  = 0
    exact_match  = 0
    per_sample   = []
    for pred, target in zip(predictions, targets):
        p    = pred.strip()
        t    = target.strip()50
        d    = Levenshtein.distance(p, t)
        cer  = (d / len(t)) if len(t) > 0 else (1.0 if len(p) > 0 else 0.0)
        total_cer   += cer
        if d != 0:
            wrong_words += 1
        else:
            exact_match += 1
        per_sample.append({
            "target": t, "pred": p,
            "cer": round(cer * 100, 2),
            "correct": d == 0
        })
    n = len(targets)
    return {
        "CER":        round((total_cer / n) * 100, 4),
        "WER":        round((wrong_words / n) * 100, 4),
        "ExactMatch": round((exact_match / n) * 100, 4),
        "N":          n,
        "per_sample": per_sample
    }


# =====================================================================
# MAIN EVALUATION FUNCTION
# =====================================================================
def run_test_evaluation():
    print("=" * 65)
    print("  HTR MODEL TEST EVALUATION")
    print("=" * 65)

    device    = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"Device     : {device}")
    print(f"Checkpoint : {TEST_CHECKPOINT}")
    print(f"Beam size  : {TEST_BEAM_SIZE}")
    print(f"Resolution : {IMG_HEIGHT}x{IMG_WIDTH}")

    # Load tokenizer
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

    # Load test data
    print("\nLoading test data...")
    test_loader, val_samples = get_test_loader(TEST_BATCH_SIZE)

    # Build model
    print("\nBuilding model...")
    model = CompleteHTRPipeline(
        vocab_size=tokenizer.vocab_size
    ).to(device)

    # Load checkpoint
    if not os.path.exists(TEST_CHECKPOINT):
        raise FileNotFoundError(
            f"Checkpoint not found: {TEST_CHECKPOINT}\n"
            "Make sure training has completed and saved a checkpoint."
        )

    ckpt = torch.load(TEST_CHECKPOINT, map_location=device,
                      weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    saved_epoch = ckpt.get('epoch', 'unknown')
    saved_wer   = ckpt.get('best_wer', 'unknown')
    print(f"Loaded checkpoint from epoch {saved_epoch} "
          f"| Saved val WER: {saved_wer:.2f}%")

    model.eval()

    # Build lexicon for verification
    agent_verifier = AgenticVerificationModule(
        words_txt_file=WORDS_TXT_FILE
    )

    # Run inference
    all_raw_preds      = []
    all_verified_preds = []
    all_targets        = []

    print(f"\nRunning beam search inference (beam={TEST_BEAM_SIZE})...")
    print("This may take a few minutes...\n")

    with torch.no_grad():
        for images, text_labels in tqdm(
                test_loader, desc="Evaluating"):
            images = images.to(device)

            # Beam search for best accuracy
            tokens = model.generate(
                images,
                max_length   = MAX_SEQ_LEN,
                bos_token_id = tokenizer.bos_token_id,
                eos_token_id = tokenizer.eos_token_id,
                beam_size    = TEST_BEAM_SIZE
            )

            for i in range(images.size(0)):
                raw      = tokenizer.decode(
                    tokens[i], skip_special_tokens=True
                ).strip()
                verified = agent_verifier.verify_and_correct(raw)
                all_raw_preds.append(raw)
                all_verified_preds.append(verified)
                all_targets.append(text_labels[i])

    # Compute metrics
    raw_m  = test_calculate_metrics(all_raw_preds, all_targets)
    ver_m  = test_calculate_metrics(all_verified_preds, all_targets)

    # Print summary
    print("\n" + "=" * 65)
    print("  RESULTS — Raw Model Output (no post-processing)")
    print("=" * 65)
    print(f"  CER        : {raw_m['CER']:.2f}%")
    print(f"  WER        : {raw_m['WER']:.2f}%")
    print(f"  Exact Match: {raw_m['ExactMatch']:.2f}%")

    print("\n" + "=" * 65)
    print("  RESULTS — After Agentic Verification")
    print("=" * 65)
    print(f"  CER        : {ver_m['CER']:.2f}%")
    print(f"  WER        : {ver_m['WER']:.2f}%")
    print(f"  Exact Match: {ver_m['ExactMatch']:.2f}%")
    print(f"  Samples    : {ver_m['N']}")

    cer_gain = round(raw_m['CER'] - ver_m['CER'], 4)
    wer_gain = round(raw_m['WER'] - ver_m['WER'], 4)
    print(f"\n  Verification CER gain : {cer_gain:+.2f}%")
    print(f"  Verification WER gain : {wer_gain:+.2f}%")

    # Sample predictions table
    print(f"\n{'=' * 65}")
    print(f"  SAMPLE PREDICTIONS (first {SHOW_SAMPLES})")
    print(f"{'=' * 65}")
    print(f"{'TARGET':<22} {'RAW PRED':<22} {'VERIFIED':<18} {'OK':>3}")
    print("-" * 65)
    for i in range(min(SHOW_SAMPLES, len(all_targets))):
        ok  = "✓" if all_verified_preds[i].strip() == \
                     all_targets[i].strip() else "✗"
        print(f"{all_targets[i]:<22} "
              f"{all_raw_preds[i]:<22} "
              f"{all_verified_preds[i]:<18} {ok:>3}")

    # Worst 10 predictions
    per   = ver_m['per_sample']
    worst = sorted(per, key=lambda x: x['cer'], reverse=True)[:10]
    print(f"\n{'=' * 65}")
    print("  TOP 10 WORST PREDICTIONS")
    print(f"{'=' * 65}")
    print(f"{'TARGET':<22} {'PREDICTED':<22} {'CER':>7}")
    print("-" * 55)
    for s in worst:
        print(f"{s['target']:<22} {s['pred']:<22} {s['cer']:>6.1f}%")

    # Length-wise CER breakdown
    print(f"\n{'=' * 65}")
    print("  CER BY WORD LENGTH")
    print(f"{'=' * 65}")
    buckets = defaultdict(list)
    for s in per:
        length = len(s['target'])
        bucket = (length // 3) * 3
        buckets[bucket].append(s['cer'])
    print(f"{'Length Range':<16} {'Samples':>8} {'Avg CER':>10}")
    print("-" * 38)
    for b in sorted(buckets.keys()):
        avg = round(sum(buckets[b]) / len(buckets[b]), 2)
        rng = f"{b}-{b+2} chars"
        print(f"{rng:<16} {len(buckets[b]):>8} {avg:>9.2f}%")

    # Save to file
    if SAVE_RESULTS:
        out_path = TEST_CHECKPOINT.replace('.pth',
                                           '_test_results.txt')
        with open(out_path, 'w') as f:
            f.write("HTR TEST RESULTS\n")
            f.write("=" * 65 + "\n")
            f.write(f"Checkpoint  : {TEST_CHECKPOINT}\n")
            f.write(f"Epoch       : {saved_epoch}\n")
            f.write(f"Beam size   : {TEST_BEAM_SIZE}\n")
            f.write(f"Resolution  : {IMG_HEIGHT}x{IMG_WIDTH}\n\n")
            f.write("Raw output:\n")
            f.write(f"  CER: {raw_m['CER']:.2f}%\n")
            f.write(f"  WER: {raw_m['WER']:.2f}%\n\n")
            f.write("After verification:\n")
            f.write(f"  CER        : {ver_m['CER']:.2f}%\n")
            f.write(f"  WER        : {ver_m['WER']:.2f}%\n")
            f.write(f"  Exact Match: {ver_m['ExactMatch']:.2f}%\n\n")
            f.write("=" * 65 + "\n")
            f.write("ALL PREDICTIONS\n")
            f.write(f"{'TARGET':<25} {'PREDICTED':<25} "
                    f"{'CER':>6} {'OK':>4}\n")
            f.write("-" * 65 + "\n")
            for tgt, pred in zip(all_targets, all_verified_preds):
                d   = Levenshtein.distance(pred.strip(), tgt.strip())
                cer = round(d / max(len(tgt), 1) * 100, 1)
                ok  = "YES" if pred.strip() == tgt.strip() else "NO"
                f.write(f"{tgt:<25} {pred:<25} "
                        f"{cer:>5.1f}% {ok:>4}\n")
        print(f"\nFull results saved to:\n{out_path}")

    return ver_m


# =====================================================================
# RUN — execute this cell
# =====================================================================
results = run_test_evaluation()

  HTR MODEL TEST EVALUATION
Device     : cuda
Checkpoint : /home/mca24-26/handwritten text detection/iam_htr_5.pth
Beam size  : 5
Resolution : 96x384

Loading test data...
Total dataset  : 38305 samples
Train split    : 34474 samples
Test split     : 3831 samples

Building model...


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded checkpoint from epoch 38 | Saved val WER: 15.51%
Lexicon built: 6231 unique words.

Running beam search inference (beam=5)...
This may take a few minutes...



Evaluating: 100%|███████████████████████████| 240/240 [1:24:10<00:00, 21.04s/it]


  RESULTS — Raw Model Output (no post-processing)
  CER        : 8.32%
  WER        : 15.77%
  Exact Match: 84.23%

  RESULTS — After Agentic Verification
  CER        : 8.39%
  WER        : 15.48%
  Exact Match: 84.52%
  Samples    : 3831

  Verification CER gain : -0.08%
  Verification WER gain : +0.29%

  SAMPLE PREDICTIONS (first 20)
TARGET                 RAW PRED               VERIFIED            OK
-----------------------------------------------------------------
any                    any                    any                  ✓
any                    any                    any                  ✓
unopposed              imposed                imposed              ✗
conference             conference             conference           ✓
Sir                    Sir                    Sir                  ✓
-                      -                      -                    ✓
the                    the                    the                  ✓
United                 United            